In [4]:
import os
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [5]:
from pathlib import Path

# Path to the dataset (notebooks folder -> project root -> data/raw)
DATASET_DIR = Path("../data/raw")

# Check if the dataset exists
if DATASET_DIR.exists():
    print(f"Dataset found at: {DATASET_DIR.resolve()}")
else:
    print("Dataset not found.")

Dataset found at: C:\Users\User\OneDrive\Desktop\Year-2-Recess-Project\data\raw


In [6]:
# Load dataset labels/metadata
CSV_PATH = DATASET_DIR / "Data_Entry_2017.csv"

df = pd.read_csv(CSV_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (112120, 12)


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,058Y,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,058Y,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,058Y,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,081Y,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,081Y,F,PA,2582,2991,0.143,0.143,NaN


In [7]:
# Check unique disease labels and their counts

label_counts = df["Finding Labels"].value_counts()

print("Number of unique labels:", len(label_counts))

label_counts.head(20)

Number of unique labels: 790


Finding Labels
No Finding                           60412
Infiltration                          9551
Atelectasis                           4212
Effusion                              3959
Nodule                                2706
Pneumothorax                          2199
Mass                                  2138
Effusion|Infiltration                 1602
Atelectasis|Infiltration              1356
Consolidation                         1314
Atelectasis|Effusion                  1167
Pleural_Thickening                    1127
Cardiomegaly                          1094
Emphysema                              895
Infiltration|Nodule                    829
Atelectasis|Effusion|Infiltration      740
Fibrosis                               727
Edema                                  634
Cardiomegaly|Effusion                  483
Consolidation|Infiltration             442
Name: count, dtype: int64

In [8]:
# Check for missing values

missing_values = df.isnull().sum()

print("Missing values per column:")
print(missing_values)

print("\nTotal missing values:", missing_values.sum())

Missing values per column:
Image Index                         0
Finding Labels                      0
Follow-up #                         0
Patient ID                          0
Patient Age                         0
Patient Gender                      0
View Position                       0
OriginalImage[Width                 0
Height]                             0
OriginalImagePixelSpacing[x         0
y]                                  0
Unnamed: 11                    112120
dtype: int64

Total missing values: 112120


In [9]:
# Remove columns that are completely empty

df = df.dropna(axis=1, how="all")

print("Dataset cleaned successfully!")
print("New shape:", df.shape)

df.head()

Dataset cleaned successfully!
New shape: (112120, 11)


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
0,00000001_000.png,Cardiomegaly,0,1,058Y,M,PA,2682,2749,0.143,0.143
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,058Y,M,PA,2894,2729,0.143,0.143
2,00000001_002.png,Cardiomegaly|Effusion,2,1,058Y,M,PA,2500,2048,0.168,0.168
3,00000002_000.png,No Finding,0,2,081Y,M,PA,2500,2048,0.171,0.171
4,00000003_000.png,Hernia,0,3,081Y,F,PA,2582,2991,0.143,0.143


In [10]:
# Check if all images in the CSV exist in the image folder

IMAGE_DIR = DATASET_DIR / "images-224"

missing_images = []

for image_name in df["Image Index"]:
    image_path = IMAGE_DIR / image_name
    
    if not image_path.exists():
        missing_images.append(image_name)

print("Total images listed in CSV:", len(df))
print("Missing image files:", len(missing_images))

if len(missing_images) > 0:
    print("Examples of missing images:")
    print(missing_images[:10])
else:
    print("All images are available!")

Total images listed in CSV: 112120
Missing image files: 112120
Examples of missing images:
['00000001_000.png', '00000001_001.png', '00000001_002.png', '00000002_000.png', '00000003_000.png', '00000003_001.png', '00000003_002.png', '00000003_003.png', '00000003_004.png', '00000003_005.png']


In [11]:
# Check the actual image folder contents

print("Image directory:", IMAGE_DIR.resolve())

# List first few items inside images-224
items = list(IMAGE_DIR.iterdir())

print("Number of items:", len(items))

print("First 10 items:")
for item in items[:10]:
    print(item)

Image directory: C:\Users\User\OneDrive\Desktop\Year-2-Recess-Project\data\raw\images-224
Number of items: 1
First 10 items:
..\data\raw\images-224\images-224


In [12]:
# Correct image directory path

IMAGE_DIR = DATASET_DIR / "images-224" / "images-224"

print("Updated image directory:", IMAGE_DIR.resolve())

# Check images
missing_images = []

for image_name in df["Image Index"]:
    image_path = IMAGE_DIR / image_name
    
    if not image_path.exists():
        missing_images.append(image_name)

print("Total images listed in CSV:", len(df))
print("Missing image files:", len(missing_images))

if len(missing_images) == 0:
    print("All images are available!")
else:
    print("Examples of missing images:")
    print(missing_images[:10])

Updated image directory: C:\Users\User\OneDrive\Desktop\Year-2-Recess-Project\data\raw\images-224\images-224
Total images listed in CSV: 112120
Missing image files: 0
All images are available!


In [13]:
# Create binary classification labels: normal vs cancerous

def assign_label(finding):
    if finding == "No Finding":
        return "normal"
    else:
        return "cancerous"

df["Label"] = df["Finding Labels"].apply(assign_label)

# Check distribution
print(df["Label"].value_counts())

Label
normal       60412
cancerous    51708
Name: count, dtype: int64


In [14]:
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temporary (val + test)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["Label"],
    random_state=42
)

# Second split: divide temporary into validation and test (10% each of total)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["Label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

print("\nTrain distribution:")
print(train_df["Label"].value_counts())

print("\nValidation distribution:")
print(val_df["Label"].value_counts())

print("\nTest distribution:")
print(test_df["Label"].value_counts())

Train size: 89696
Validation size: 11212
Test size: 11212

Train distribution:
Label
normal       48330
cancerous    41366
Name: count, dtype: int64

Validation distribution:
Label
normal       6041
cancerous    5171
Name: count, dtype: int64

Test distribution:
Label
normal       6041
cancerous    5171
Name: count, dtype: int64


In [15]:
id="copy_dataset_folders"
from pathlib import Path
import shutil

# Destination folders
OUTPUT_DIR = DATASET_DIR

splits = {
    "train": train_df,
    "val": val_df,
    "test": test_df
}

# Create folders and copy images
for split_name, split_df in splits.items():
    for label in ["normal", "cancerous"]:
        
        # Create folder
        dest_dir = OUTPUT_DIR / split_name / label
        dest_dir.mkdir(parents=True, exist_ok=True)
        
        # Filter images
        label_df = split_df[split_df["Label"] == label]
        
        print(f"Copying {split_name}/{label}: {len(label_df)} images")
        
        for image_name in label_df["Image Index"]:
            src = IMAGE_DIR / image_name
            dst = dest_dir / image_name
            
            if not dst.exists():
                shutil.copy2(src, dst)

print("Dataset organization completed!")

Copying train/normal: 48330 images
Copying train/cancerous: 41366 images
Copying val/normal: 6041 images
Copying val/cancerous: 5171 images
Copying test/normal: 6041 images
Copying test/cancerous: 5171 images
Dataset organization completed!


In [16]:
# Create metadata CSV

metadata = pd.concat([
    train_df.assign(Split="train"),
    val_df.assign(Split="val"),
    test_df.assign(Split="test")
])

# Add full image paths
metadata["File Path"] = metadata.apply(
    lambda row: str(Path(row["Split"]) / row["Label"] / row["Image Index"]),
    axis=1
)

# Select useful columns
metadata = metadata[
    [
        "File Path",
        "Image Index",
        "Finding Labels",
        "Label",
        "Split"
    ]
]

# Save metadata
METADATA_PATH = DATASET_DIR / "metadata.csv"

metadata.to_csv(METADATA_PATH, index=False)

print("Metadata created successfully!")
print("Saved at:", METADATA_PATH.resolve())

metadata.head()

Metadata created successfully!
Saved at: C:\Users\User\OneDrive\Desktop\Year-2-Recess-Project\data\raw\metadata.csv


,File Path,Image Index,Finding Labels,Label,Split
58641,train\cancerous\00014520_017.png,00014520_017.png,Infiltration,cancerous,train
79386,train\normal\00019496_000.png,00019496_000.png,No Finding,normal,train
60066,train\cancerous\00014822_041.png,00014822_041.png,Effusion|Pleural_Thickening|Pneumothorax,cancerous,train
70998,train\normal\00017510_000.png,00017510_000.png,No Finding,normal,train
19872,train\cancerous\00005298_005.png,00005298_005.png,Infiltration|Pleural_Thickening,cancerous,train


In [17]:
# Final dataset structure verification

from pathlib import Path

folders = [
    DATASET_DIR / "train" / "normal",
    DATASET_DIR / "train" / "cancerous",
    DATASET_DIR / "val" / "normal",
    DATASET_DIR / "val" / "cancerous",
    DATASET_DIR / "test" / "normal",
    DATASET_DIR / "test" / "cancerous",
]

print("Dataset folder verification:\n")

for folder in folders:
    count = len(list(folder.glob("*.png")))
    print(f"{folder.relative_to(DATASET_DIR)} : {count} images")

print("\nMetadata file exists:", (DATASET_DIR / "metadata.csv").exists())

print("\nTotal images organized:",
      sum(len(list(folder.glob("*.png"))) for folder in folders))

Dataset folder verification:

train\normal : 48330 images
train\cancerous : 41366 images
val\normal : 6041 images
val\cancerous : 5171 images
test\normal : 6041 images
test\cancerous : 5171 images

Metadata file exists: True

Total images organized: 112120


In [18]:
from pathlib import Path

# Define dataset path again
DATASET_DIR = Path("../data/raw")

print("Dataset directory:", DATASET_DIR.resolve())

Dataset directory: C:\Users\User\OneDrive\Desktop\Year-2-Recess-Project\data\raw


In [19]:
from pathlib import Path

DOC_PATH = DATASET_DIR / "dataset_info.md"

content = f"""# Chest X-Ray Dataset Information

## Dataset Source
NIH Chest X-ray Dataset (NIH Clinical Center)

## Dataset Description
The dataset contains 112,120 chest X-ray images with associated disease labels.

## Dataset Organization
- train/normal: {len(list((DATASET_DIR / "train" / "normal").glob("*.png")))} images
- train/cancerous: {len(list((DATASET_DIR / "train" / "cancerous").glob("*.png")))} images
- val/normal: {len(list((DATASET_DIR / "val" / "normal").glob("*.png")))} images
- val/cancerous: {len(list((DATASET_DIR / "val" / "cancerous").glob("*.png")))} images
- test/normal: {len(list((DATASET_DIR / "test" / "normal").glob("*.png")))} images
- test/cancerous: {len(list((DATASET_DIR / "test" / "cancerous").glob("*.png")))} images

## Files
- Labels CSV: {CSV_PATH}
- Metadata CSV: {METADATA_PATH}
"""

DOC_PATH.write_text(content, encoding="utf-8")
print(f"Dataset info file created at: {DOC_PATH.resolve()}")


Dataset info file created at: C:\Users\User\OneDrive\Desktop\Year-2-Recess-Project\data\raw\dataset_info.md
